In [0]:
from pyspark.sql import functions as F

# Read the taxi table and print basic info: row count, column count, schema, and summary stats
df = spark.read.table("group3_gp.bronze.high_volume_fhv")
print(f"Row count : {df.count():,}")
print(f"Columns   : {len(df.columns)}")
df.printSchema()


In [0]:
%sql
-- ════════════════════════════════════════════════
-- HIGH VOLUME FHV — Bad data checks
-- Fare columns ONLY available from ~2021 onwards
-- Nulls in fare columns pre-2021 are EXPECTED
-- ════════════════════════════════════════════════

-- Total row count
SELECT 'Total rows' AS check_name, COUNT(*) AS result
FROM group3_gp.testing.high_volume_fhv

UNION ALL

-- Trips where dropoff is before or equal to pickup
SELECT 'Bad timestamps (dropoff <= pickup)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.high_volume_fhv
WHERE TO_TIMESTAMP(dropoff_datetime) <= TO_TIMESTAMP(pickup_datetime)

UNION ALL

-- Zero or negative trip miles
SELECT 'Bad distances (trip_miles <= 0)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.high_volume_fhv
WHERE CAST(trip_miles AS DOUBLE) <= 0

UNION ALL

-- Only check fares where the column IS populated — nulls pre-2021 are expected
SELECT 'Bad fares (populated but <= 0)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.high_volume_fhv
WHERE base_passenger_fare IS NOT NULL
  AND CAST(base_passenger_fare AS DOUBLE) <= 0

UNION ALL

-- Zero or negative driver pay where column is populated
SELECT 'Bad driver_pay (populated but <= 0)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.high_volume_fhv
WHERE driver_pay IS NOT NULL
  AND CAST(driver_pay AS DOUBLE) <= 0

In [0]:
%sql
-- HVFHV — Company breakdown
-- Maps hvfhs_license_num to company names
SELECT
    hvfhs_license_num,
    CASE hvfhs_license_num
        WHEN 'HV0002' THEN 'Juno'
        WHEN 'HV0003' THEN 'Uber'
        WHEN 'HV0004' THEN 'Via'
        WHEN 'HV0005' THEN 'Lyft'
        ELSE 'Unknown'
    END AS company,
    COUNT(*) AS trip_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM group3_gp.testing.high_volume_fhv
GROUP BY hvfhs_license_num
ORDER BY trip_count DESC

In [0]:
%sql
-- HVFHV — Fare availability by year
-- Confirms nulls pre-2021 are expected behaviour
SELECT
    Year,
    COUNT(*)                    AS total_rows,
    COUNT(base_passenger_fare)  AS fare_populated,
    COUNT(driver_pay)           AS driver_pay_populated,
    ROUND(
        COUNT(base_passenger_fare) * 100.0 / COUNT(*), 1
    )                           AS fare_coverage_pct
FROM group3_gp.testing.high_volume_fhv
GROUP BY Year
ORDER BY Year

In [0]:
%sql
-- HVFHV — Shared ride breakdown by company
-- NOTE: Lyft (HV0005) overcounts — flags requested-but-unmatched rides too
SELECT
    CASE hvfhs_license_num
        WHEN 'HV0002' THEN 'Juno'
        WHEN 'HV0003' THEN 'Uber'
        WHEN 'HV0004' THEN 'Via'
        WHEN 'HV0005' THEN 'Lyft (overcounts)'
        ELSE 'Unknown'
    END                  AS company,
    shared_request_flag,
    shared_match_flag,
    COUNT(*)             AS trip_count
FROM group3_gp.testing.high_volume_fhv
GROUP BY hvfhs_license_num, shared_request_flag, shared_match_flag
ORDER BY company, trip_count DESC

In [0]:
%sql
-- HVFHV — WAV (wheelchair-accessible vehicle) trips
-- Shows request vs match rates for accessibility compliance
SELECT
    wav_request_flag,
    wav_match_flag,
    COUNT(*) AS trip_count
FROM group3_gp.testing.high_volume_fhv
GROUP BY wav_request_flag, wav_match_flag
ORDER BY trip_count DESC

In [0]:
%sql
-- HVFHV — Year distribution
-- Shows trip volume per year to spot coverage gaps
SELECT Year, COUNT(*) AS trip_count
FROM group3_gp.testing.high_volume_fhv
GROUP BY Year
ORDER BY Year